# Task 6 Phase 2 — Gateway Hub Selection

Apply a single global Set-Cover MIP to select gateway hubs across all 132 freight areas.
The formulation mirrors Task 5's regional hub MIP but operates at the area tier with
demand-scaled coverage targets and a co-area separation constraint.

**Subtasks implemented here:**
- **6.7** — Build gateway candidate set G and feasibility set Z_gw
- **6.8** — Pre-compute objective coefficients ĉ_ga and capacity parameters
- **6.9** — Build and solve the GatewayMIP; extract raw solution
- **6.10** — Characterise selected gateways and assess solution quality
- **6.11** — Define area-to-hub and inter-area network links
- **6.12** — Produce figures and tabular exports

In [ ]:
# ── Imports and paths ────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')

import sys
import math
import numpy as np
import pandas as pd
import geopandas as gpd
from pyproj import Transformer
from pathlib import Path

# Locate project root
ROOT = Path('.').resolve()
for _ in range(4):
    if (ROOT / 'Data').exists() and (ROOT / 'Doc').exists():
        break
    ROOT = ROOT.parent

# Add Task6/ to path so we can import gateway_mip_solver
TASK6_DIR = ROOT / 'Task6'
if str(TASK6_DIR) not in sys.path:
    sys.path.insert(0, str(TASK6_DIR))

DATA_T3   = ROOT / 'Data/Task3'
DATA_T4   = ROOT / 'Data/Task4/processed'
DATA_T5   = ROOT / 'Data/Task5'
DATA_T6   = ROOT / 'Data/Task6'
CACHE     = DATA_T6 / 'cache'
FIG_DIR   = DATA_T6 / 'figures'
CACHE.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

print(f'Project root : {ROOT}')
print(f'Cache dir    : {CACHE}')
print(f'Figures dir  : {FIG_DIR}')

---

## 6.7 — Candidate Set G and Z_gw Construction

Build G (gateway candidate pool), the m_a targets, the feasibility set Z_gw,
and the separation clash set S.

### Candidate pool G

$$G = \{g \in \text{CoStar} : s_g \geq 20{,}000 \text{ sqft},\; g \notin \text{Task 5 selected hubs}\}$$

### m_a — demand-scaled gateway count per area

$$m_a = \max\!\left(1,\; \min\!\left(5,\; \left\lceil \frac{2 \cdot T_a}{\bar{T}}\right\rceil\right)\right)$$

where $\bar{T}$ is the mean external throughput across all 132 areas.

### Z_gw feasibility

$$(g, a) \in Z_{gw} \iff \text{dist}(g, \text{centroid}_a) \leq 80{,}467\,\text{m (50 mi)}\;\text{ OR }\;\text{county}(g) \in \text{area}\,a$$

### Separation clash set S

$$S = \{(g, g', a) : (g,a) \in Z_{gw},\; (g',a) \in Z_{gw},\; g < g',\; \text{dist}(g,g') < 32{,}187\,\text{m (20 mi)}\}$$

In [ ]:
# ── 6.7.1  Load candidate pool and exclude Task 5 regional hubs ──────────────
DIST_COVER_M    = 80_467.0   # 50 miles in meters
DIST_SEP_M      = 32_187.0   # 20 miles in meters (separation threshold)
MIN_SPACE_SQ_FT = 20_000     # gateway size floor

# Load full CoStar preprocessed pool
all_cands = pd.read_csv(DATA_T4 / 'preprocessed_capacity_location.csv', dtype={'county_fips': str})
all_cands['county_fips'] = all_cands['county_fips'].str.zfill(5)
print(f'Raw CoStar pool        : {len(all_cands):,} rows')

# Apply ≥ 20k sqft filter
G_raw = all_cands[all_cands['usable_available_space_sf'] >= MIN_SPACE_SQ_FT].copy()
print(f'After ≥20k sqft filter : {len(G_raw):,} rows')

# Load Task 5 selected hubs and exclude them
t5_hubs = pd.read_csv(DATA_T5 / 'selected_hubs.csv')
excluded_ids = set(t5_hubs['candidate_id'].astype(str))
G_raw = G_raw[~G_raw['candidate_id'].astype(str).isin(excluded_ids)].copy()
n_excluded = len(all_cands) - len(all_cands[all_cands['usable_available_space_sf'] >= MIN_SPACE_SQ_FT]) + \
             (len(all_cands[all_cands['usable_available_space_sf'] >= MIN_SPACE_SQ_FT]) - len(G_raw))
print(f'Task 5 hubs excluded   : {len(excluded_ids) - (len(all_cands) - len(all_cands[all_cands["usable_available_space_sf"] >= MIN_SPACE_SQ_FT])):,}')
print(f'Final |G|              : {len(G_raw):,} candidates')

In [ ]:
# ── 6.7.2  Project G to EPSG:9311 ────────────────────────────────────────────
transformer = Transformer.from_crs('EPSG:4326', 'EPSG:9311', always_xy=True)
G_raw = G_raw.reset_index(drop=True)
G_raw['g_idx'] = G_raw.index

x_m, y_m = transformer.transform(G_raw['longitude'].values, G_raw['latitude'].values)
G_raw['x_m'] = x_m
G_raw['y_m'] = y_m

print(f'Projected {len(G_raw):,} gateway candidates to EPSG:9311')
print(f'x_m range: [{G_raw["x_m"].min():,.0f}, {G_raw["x_m"].max():,.0f}]')
print(f'y_m range: [{G_raw["y_m"].min():,.0f}, {G_raw["y_m"].max():,.0f}]')
print(f'sqft range: [{G_raw["usable_available_space_sf"].min():,.0f}, {G_raw["usable_available_space_sf"].max():,.0f}]')
print()
print('Per-state candidate count:')
print(G_raw['source_state'].value_counts().to_string())

In [ ]:
# ── 6.7.3  Compute m_a — demand-scaled gateway count per area ────────────────
area_metrics = pd.read_csv(DATA_T6 / 'area_metrics.csv')
area_ids     = area_metrics['area_id'].values            # string labels like '0_0'
n_areas      = len(area_metrics)

# DO NOT hard-code T_mean; compute at runtime
T_mean = float(area_metrics['external_throughput_ktons'].mean())

def compute_m_a(T_a: float, T_mean: float) -> int:
    """Demand-scaled gateway count: max(1, min(5, ceil(2*T_a / T_mean)))."""
    return max(1, min(5, math.ceil(2.0 * T_a / T_mean)))

area_metrics['m_a'] = area_metrics['external_throughput_ktons'].apply(
    lambda T: compute_m_a(T, T_mean)
)

m_a = area_metrics['m_a'].values.astype(np.int32)   # shape (132,)

print(f'T_mean (runtime) = {T_mean:,.1f} ktons')
print(f'Σ m_a            = {m_a.sum():,} (expected ~319)')
print(f'mean m_a         = {m_a.mean():.2f} (expected ~2.42)')
print()
print('m_a distribution:')
print(pd.Series(m_a).value_counts().sort_index())

In [ ]:
# ── 6.7.4  Build Z_gw (centroid radius + in-area join) ───────────────────────
# Strategy:
#   1. Euclidean distance from each g to each area centroid in EPSG:9311.
#   2. Include (g, a) if dist ≤ 80,467 m.
#   3. Also include (g, a) if county_fips of g belongs to area a (in-area).

Z_gw_cache = CACHE / 'Z_gw_pairs.npy'

if Z_gw_cache.exists():
    Z_gw = np.load(Z_gw_cache)   # shape (|Z_gw|, 2) int32
    print(f'Loaded Z_gw from cache: {len(Z_gw):,} pairs')
else:
    # Gateway coords array
    G_x = G_raw['x_m'].values   # (|G|,)
    G_y = G_raw['y_m'].values   # (|G|,)

    # Area centroid arrays
    A_cx = area_metrics['centroid_x_m'].values   # (|A|,)
    A_cy = area_metrics['centroid_y_m'].values   # (|A|,)

    # Pairwise Euclidean distances via broadcasting (|G| x |A|)
    dx = G_x[:, None] - A_cx[None, :]   # (|G|, |A|)
    dy = G_y[:, None] - A_cy[None, :]   # (|G|, |A|)
    dist_mat = np.sqrt(dx**2 + dy**2)   # (|G|, |A|)

    # Centroid-radius pairs
    radius_g, radius_a = np.where(dist_mat <= DIST_COVER_M)
    radius_set = set(zip(radius_g.tolist(), radius_a.tolist()))
    print(f'Centroid-radius pairs (≤50 mi) : {len(radius_set):,}')

    # In-area pairs: join G.county_fips → area_assignment.fips → area_id
    area_assign = pd.read_csv(DATA_T6 / 'area_assignment.csv', dtype={'fips': str})
    area_assign['fips'] = area_assign['fips'].str.zfill(5)

    # Map area_id string → a_idx integer
    area_id_to_idx = {aid: i for i, aid in enumerate(area_ids)}

    fips_to_area_idx = (
        area_assign.set_index('fips')['area_id']
        .map(area_id_to_idx)
        .dropna()
        .astype(int)
        .to_dict()
    )

    # For each gateway, look up its county's area
    in_area_set = set()
    for g_idx, fips in enumerate(G_raw['county_fips'].values):
        a_idx = fips_to_area_idx.get(str(fips).zfill(5))
        if a_idx is not None:
            in_area_set.add((g_idx, a_idx))
    print(f'In-area pairs                  : {len(in_area_set):,}')

    # Union
    all_pairs = radius_set | in_area_set
    Z_gw = np.array(sorted(all_pairs), dtype=np.int32)   # (|Z_gw|, 2)
    np.save(Z_gw_cache, Z_gw)
    print(f'|Z_gw| total (union)           : {len(Z_gw):,} pairs  → saved')

In [ ]:
# ── 6.7.5  Verify every area has ≥ m_a entries in Z_gw ──────────────────────
from collections import defaultdict

z_by_a: dict[int, list] = defaultdict(list)
for z_pos, (g, a) in enumerate(Z_gw):
    z_by_a[int(a)].append(int(g))

coverage_ok = True
for a_idx in range(len(area_metrics)):
    n_in_z   = len(z_by_a[a_idx])
    target   = int(m_a[a_idx])
    if n_in_z < target:
        print(f'  WARNING area {area_ids[a_idx]}: only {n_in_z} candidates in Z_gw, m_a={target}')
        coverage_ok = False

if coverage_ok:
    print('All 132 areas have ≥ m_a candidates in Z_gw  ✓')
else:
    print('Some areas fall short — see warnings above')

print()
print('Z_gw candidates-per-area distribution:')
counts_per_area = pd.Series({area_ids[a]: len(z_by_a[a]) for a in range(len(area_metrics))})
print(counts_per_area.describe().apply(lambda x: f'{x:.1f}'))

# Log Appalachian sparse areas (regions 4 and 16)
sparse_areas = area_metrics[area_metrics['region_id'].isin([4, 16])]['area_id'].tolist()
print(f'\nAppalachian areas (regions 4 & 16): {sparse_areas}')
for aid in sparse_areas:
    a_idx = list(area_ids).index(aid)
    print(f'  {aid}: n_candidates={len(z_by_a[a_idx])}, m_a={m_a[a_idx]}')

In [ ]:
# ── 6.7.6  Build separation clash set S ─────────────────────────────────────
sep_cache = CACHE / 'separation_clashes.parquet'

if sep_cache.exists():
    sep_df = pd.read_parquet(sep_cache)
    S = list(zip(sep_df['g_idx'].tolist(), sep_df['g_prime_idx'].tolist(), sep_df['a_idx'].tolist()))
    print(f'Loaded separation clashes from cache: |S| = {len(S):,}')
else:
    G_x = G_raw['x_m'].values
    G_y = G_raw['y_m'].values

    S_rows = []
    for a_idx in range(len(area_metrics)):
        g_list = sorted(z_by_a[a_idx])
        n = len(g_list)
        if n < 2:
            continue
        # Vectorised pairwise distance for this area's gateway subset
        g_x_sub = G_x[g_list]
        g_y_sub = G_y[g_list]
        dx = g_x_sub[:, None] - g_x_sub[None, :]   # (n, n)
        dy = g_y_sub[:, None] - g_y_sub[None, :]   # (n, n)
        d  = np.sqrt(dx**2 + dy**2)                 # (n, n)
        # Upper-triangle pairs with dist < 20 mi threshold
        ii, jj = np.where((d < DIST_SEP_M) & (np.arange(n)[:, None] < np.arange(n)[None, :]))
        for i, j in zip(ii.tolist(), jj.tolist()):
            S_rows.append((g_list[i], g_list[j], a_idx))

    S = S_rows
    sep_df = pd.DataFrame(S, columns=['g_idx', 'g_prime_idx', 'a_idx'])
    sep_df.to_parquet(sep_cache, index=False)
    print(f'Built separation clash set: |S| = {len(S):,}  → saved')

if len(S) > 500_000:
    print(f'NOTE: |S|={len(S):,} > 500,000 — consider lazy constraint addition')
else:
    print(f'|S| = {len(S):,} — all constraints will be added upfront')

In [ ]:
# ── 6.7.7  Save G_candidates and area_metrics_phase2 ────────────────────────
G_cache = CACHE / 'G_candidates.parquet'
G_raw.to_parquet(G_cache, index=False)
print(f'G_candidates saved  : {G_cache}  ({len(G_raw):,} rows)')

# Save area_metrics with m_a
area_metrics.to_csv(DATA_T6 / 'area_metrics_phase2.csv', index=False)
print(f'area_metrics_phase2 : {DATA_T6 / "area_metrics_phase2.csv"}')

# ── 6.7 Sanity checks ────────────────────────────────────────────────────────
assert len(G_raw) > 0,                    'G is empty'
assert len(G_raw) < len(all_cands),       'No Task 5 hubs excluded'
assert Z_gw.shape[1] == 2,               'Z_gw must be (n, 2)'
assert Z_gw[:, 0].max() < len(G_raw),    'g_idx out of bounds'
assert Z_gw[:, 1].max() < n_areas,       'a_idx out of bounds'
assert m_a.min() >= 1,                   'm_a must be ≥ 1'
assert m_a.max() <= 5,                   'm_a must be ≤ 5'
assert m_a.sum() > 200,                  'Σ m_a suspiciously low'

print()
print('6.7 sanity checks passed  ✓')
print(f'  |G|      = {len(G_raw):,}')
print(f'  |Z_gw|   = {len(Z_gw):,}')
print(f'  |S|      = {len(S):,}')
print(f'  Σ m_a    = {m_a.sum():,}')

---

## 6.8 — Objective Coefficients and Capacity Parameters

### Objective cost coefficient ĉ_ga

$$\hat{c}_{ga} = \left(\sum_{c \in C_a} w_{ca} \cdot d_{gca}\right) \cdot \left(\frac{\bar{Q}_{gw}}{s_g}\right)^{0.5}$$

where $w_{ca}$ is county throughput (ktons), $d_{gca}$ is Euclidean distance from
gateway $g$ to county centroid $c$ (meters), and $\bar{Q}_{gw}$ is the median sqft
across G.

### Capacity constraint RHS

$$\text{RHS}_a = \bar{Q}_{gw} \cdot \frac{T_a}{\bar{T}}$$

In [ ]:
# ── 6.8.1  Load county data and area assignment ────────────────────────────────
# Reload in case of kernel restart
G_raw        = pd.read_parquet(CACHE / 'G_candidates.parquet')
Z_gw         = np.load(CACHE / 'Z_gw_pairs.npy')
area_metrics = pd.read_csv(DATA_T6 / 'area_metrics_phase2.csv')
area_ids     = area_metrics['area_id'].values
m_a          = area_metrics['m_a'].values.astype(np.int32)

sep_df       = pd.read_parquet(CACHE / 'separation_clashes.parquet')
S            = list(zip(sep_df['g_idx'].tolist(), sep_df['g_prime_idx'].tolist(), sep_df['a_idx'].tolist()))

# County geometry + throughput weights
counties = gpd.read_file(DATA_T3 / 'derived/ne_counties_prepared.gpkg')
counties['fips'] = counties['fips'].astype(str).str.zfill(5)
# w_ca = county throughput (ktons) used as demand weight
counties['w'] = counties['throughput'].fillna(0.0).clip(lower=1e-6)

# Area assignment: fips → area_id → a_idx
area_assign = pd.read_csv(DATA_T6 / 'area_assignment.csv', dtype={'fips': str})
area_assign['fips'] = area_assign['fips'].str.zfill(5)
area_id_to_idx = {aid: i for i, aid in enumerate(area_ids)}
area_assign['a_idx'] = area_assign['area_id'].map(area_id_to_idx)

# Merge a_idx and weight onto county layer
counties = counties.merge(area_assign[['fips', 'a_idx']], on='fips', how='left')

print(f'Counties loaded : {len(counties):,}  (unmapped: {counties["a_idx"].isna().sum()})')
print(f'Z_gw pairs      : {len(Z_gw):,}')

In [ ]:
# ── 6.8.2  Scalar parameters Q̄_gw and T̄ ─────────────────────────────────────
s_g    = G_raw['usable_available_space_sf'].values   # shape (|G|,)
Q_bar  = float(np.median(s_g))                       # median sqft across G
T_bar  = float(area_metrics['external_throughput_ktons'].mean())  # mean T_a
T_a    = area_metrics['external_throughput_ktons'].values          # shape (|A|,)

print(f'Q̄_gw (median sqft)        = {Q_bar:,.0f} sqft  (EDA-expected ≈296,862)')
print(f'T̄   (mean area throughput) = {T_bar:,.1f} ktons  (EDA-expected ≈3,389)')

disc_min = (Q_bar / s_g.max())**0.5
disc_max = (Q_bar / s_g.min())**0.5
print(f'\nCapacity discount factor range:')
print(f'  min (largest gw, s={s_g.max():,.0f}): {disc_min:.3f}  ({(1-disc_min)*100:.1f}% discount)')
print(f'  max (smallest gw, s={s_g.min():,.0f}): {disc_max:.3f}  ({(disc_max-1)*100:.1f}% premium)')

In [ ]:
# ── 6.8.3  Compute ĉ_ga for all (g, a) ∈ Z_gw ────────────────────────────────
c_hat_cache = CACHE / 'c_hat.npy'

if c_hat_cache.exists():
    c_hat = np.load(c_hat_cache)
    print(f'Loaded c_hat from cache: {len(c_hat):,} values')
else:
    # Build per-area county lookup: a_idx → (coords (n_c,2), weights (n_c,))
    county_by_area: dict[int, dict] = {}
    for a_idx_val in range(len(area_metrics)):
        mask = counties['a_idx'] == a_idx_val
        sub  = counties[mask]
        county_by_area[a_idx_val] = {
            'coords': sub[['centroid_x', 'centroid_y']].values.astype(np.float64),
            'w':      sub['w'].values.astype(np.float64),
        }

    # Gateway coord and sqft arrays
    G_x_arr = G_raw['x_m'].values    # (|G|,)
    G_y_arr = G_raw['y_m'].values    # (|G|,)
    s_g_arr = G_raw['usable_available_space_sf'].values  # (|G|,)

    # Group Z_gw rows by a_idx for batch computation
    from collections import defaultdict
    z_by_a_local: dict[int, list] = defaultdict(list)
    for z_pos, (g, a) in enumerate(Z_gw):
        z_by_a_local[int(a)].append((z_pos, int(g)))

    c_hat = np.zeros(len(Z_gw), dtype=np.float64)

    for a_idx_val, z_list in z_by_a_local.items():
        area_data = county_by_area[a_idx_val]
        c_coords  = area_data['coords']   # (n_c, 2)
        c_w       = area_data['w']        # (n_c,)

        if len(c_coords) == 0:
            # No counties mapped to this area — use 0 weight
            continue

        # Batch: distances from all gateways in this area's Z to all counties
        g_indices = [g for _, g in z_list]
        g_x_batch = G_x_arr[g_indices]   # (n_g_batch,)
        g_y_batch = G_y_arr[g_indices]   # (n_g_batch,)
        s_batch   = s_g_arr[g_indices]   # (n_g_batch,)

        # (n_g_batch, n_c) distance matrix
        dx = g_x_batch[:, None] - c_coords[None, :, 0]   # (n_g, n_c)
        dy = g_y_batch[:, None] - c_coords[None, :, 1]   # (n_g, n_c)
        dist_gc = np.sqrt(dx**2 + dy**2)                  # (n_g, n_c)

        # Weighted sum: ∑_c w_c · d_gca  →  (n_g,)
        geo_cost = (dist_gc * c_w[None, :]).sum(axis=1)

        # Capacity discount factor: (Q̄ / s_g)^0.5
        cap_factor = (Q_bar / np.maximum(s_batch, 1.0))**0.5

        # Write into c_hat at the correct Z positions
        for batch_pos, (z_pos, _) in enumerate(z_list):
            c_hat[z_pos] = geo_cost[batch_pos] * cap_factor[batch_pos]

    np.save(c_hat_cache, c_hat)
    print(f'Computed c_hat for {len(c_hat):,} (g, a) pairs  → saved')

print(f'c_hat stats: min={c_hat.min():.2e}, median={np.median(c_hat):.2e}, max={c_hat.max():.2e}')

In [ ]:
# ── 6.8.4  Capacity constraint RHS ───────────────────────────────────────────
# RHS_a = Q̄_gw × (T_a / T̄)
cap_rhs = Q_bar * (T_a / T_bar)   # shape (|A|,)

print('Capacity RHS distribution (sqft):')
print(pd.Series(cap_rhs, index=area_ids).describe().apply(lambda x: f'{x:,.0f}'))

# Feasibility check: each area's Z_gw must have enough total capacity
from collections import defaultdict as _dd
z_by_a_cap: dict[int, list] = _dd(list)
for z_pos, (g, a) in enumerate(Z_gw):
    z_by_a_cap[int(a)].append(int(g))

cap_fail = 0
for a_idx_val in range(len(area_metrics)):
    g_list_cap = z_by_a_cap[a_idx_val]
    total_cap  = s_g[g_list_cap].sum() if g_list_cap else 0.0
    if total_cap < cap_rhs[a_idx_val]:
        print(f'  WARNING area {area_ids[a_idx_val]}: total_cap={total_cap:,.0f} < RHS={cap_rhs[a_idx_val]:,.0f}')
        cap_fail += 1

if cap_fail == 0:
    print('\nAll 132 areas have sufficient Z_gw capacity vs RHS  ✓')
else:
    print(f'\n{cap_fail} area(s) fail capacity check — consider expanding Z_gw radius')

# Save capacity RHS
np.save(CACHE / 'cap_rhs.npy', cap_rhs)
np.save(CACHE / 's_g.npy',     s_g)
print(f'\nSaved cap_rhs ({len(cap_rhs)}) and s_g ({len(s_g)}) to cache')

In [ ]:
# ── 6.8 Sanity checks ────────────────────────────────────────────────────────
assert len(c_hat) == len(Z_gw),           'c_hat length must equal |Z_gw|'
assert np.all(np.isfinite(c_hat)),        'c_hat contains NaN or inf'
assert np.all(c_hat >= 0),               'c_hat must be non-negative'
assert len(cap_rhs) == len(area_metrics), 'cap_rhs length must equal |A|'
assert np.all(cap_rhs > 0),              'cap_rhs must be positive'

print('6.8 sanity checks passed  ✓')
print(f'  Q̄_gw   = {Q_bar:,.0f} sqft')
print(f'  T̄       = {T_bar:,.1f} ktons')
print(f'  RHS_a range: [{cap_rhs.min():,.0f}, {cap_rhs.max():,.0f}] sqft')
print(f'  c_hat range: [{c_hat.min():.2e}, {c_hat.max():.2e}]')

---

## 6.9 — MIP Build and Gurobi Solution

$$\min \sum_{(g,a) \in Z_{gw}} \hat{c}_{ga} \cdot A_{ga}$$

Subject to (132 areas, 2,000+ gateway candidates):

| # | Constraint |
|---|------------|
| 1 | $\sum_g A_{ga} \geq m_a$   ∀a  (coverage) |
| 2 | $\sum_a A_{ga} \leq 2$     ∀g  (concentration cap) |
| 3 | $O_g \leq \sum_a A_{ga}$  ∀g  (activation) |
| 4 | $\sum_g A_{ga} s_g \geq \text{RHS}_a$  ∀a  (capacity) |
| 5 | $A_{ga} + A_{g'a} \leq 1$  ∀(g,g',a) ∈ S  (separation ≥ 20 mi) |

In [ ]:
# ── 6.9.1  Load all cached inputs ────────────────────────────────────────────
from gateway_mip_solver import GatewayMIP, GatewayMIPParameters

G_raw        = pd.read_parquet(CACHE / 'G_candidates.parquet')
Z_gw         = np.load(CACHE / 'Z_gw_pairs.npy')
c_hat        = np.load(CACHE / 'c_hat.npy')
cap_rhs      = np.load(CACHE / 'cap_rhs.npy')
s_g          = np.load(CACHE / 's_g.npy')
area_metrics = pd.read_csv(DATA_T6 / 'area_metrics_phase2.csv')
area_ids     = area_metrics['area_id'].values
m_a          = area_metrics['m_a'].values.astype(np.int32)

sep_df       = pd.read_parquet(CACHE / 'separation_clashes.parquet')
S            = list(zip(sep_df['g_idx'].tolist(), sep_df['g_prime_idx'].tolist(), sep_df['a_idx'].tolist()))

G_arr = np.arange(len(G_raw), dtype=np.int64)

print(f'|G|     = {len(G_raw):,}')
print(f'|A|     = {len(area_metrics):,}')
print(f'|Z_gw|  = {len(Z_gw):,}')
print(f'|S|     = {len(S):,}')
print(f'Σ m_a   = {m_a.sum():,}')

In [ ]:
# ── 6.9.2  Build the Gurobi model ────────────────────────────────────────────
params = GatewayMIPParameters(
    time_limit        = 600,   # 10-minute wall-clock limit
    mip_gap           = 0.01,  # stop at 1% optimality gap
    output_flag       = 1,     # print Gurobi log
    concentration_cap = 2,     # each gateway serves ≤ 2 areas
    threads           = 0,     # use all available cores
)

solver = GatewayMIP(
    G_arr   = G_arr,
    Z       = Z_gw,
    c_hat   = c_hat,
    m_a     = m_a,
    cap_rhs = cap_rhs,
    s_g     = s_g,
    S       = S,
    params  = params,
)

solver.build()
print(solver)

In [ ]:
# ── 6.9.3  Solve ──────────────────────────────────────────────────────────────
result = solver.solve()

print(f"\n{'═'*55}")
print(f"  Gurobi status    : {result.status_name}")
print(f"  Objective value  : {result.objective:,.0f}")
print(f"  MIP gap          : {result.mip_gap*100:.3f} %")
print(f"  Solve time       : {result.solve_time:.1f} s")
print(f"  Open gateways    : {result.n_gateways_open:,}")
print(f"  Assignments      : {result.n_assignments:,}")
print(f"{'═'*55}")

if not result.is_feasible():
    print('\nMIP returned no feasible solution — check infeasible areas and relax constraint 1')
    raise RuntimeError('GatewayMIP infeasible')

In [ ]:
# ── 6.9.4  Extract solution DataFrames ───────────────────────────────────────
selected_gw_df, assignments_df = solver.extract_solution(G_raw, area_ids)

print(f'Selected gateways : {len(selected_gw_df):,}')
print(f'Assignments       : {len(assignments_df):,}')
print()

# Concentration summary
conc = selected_gw_df['n_areas_served'].value_counts().sort_index()
for n_areas_srv, count in conc.items():
    print(f'  Gateways serving {n_areas_srv} area(s): {count:,}')

print()
print(selected_gw_df[['candidate_id','facility_name','city','source_state',
                        'usable_available_space_sf','n_areas_served']].head(8).to_string(index=False))

In [ ]:
# ── 6.9.5  Save raw solution ─────────────────────────────────────────────────
selected_gw_df.to_parquet(CACHE / 'selected_gw.parquet', index=False)
assignments_df.to_parquet(CACHE / 'assignments.parquet',  index=False)

# Save solve metadata
import json as _json
solve_meta = {
    'status':          result.status_name,
    'objective':       result.objective,
    'mip_gap':         result.mip_gap,
    'solve_time_s':    result.solve_time,
    'n_gateways_open': result.n_gateways_open,
    'n_assignments':   result.n_assignments,
}
with open(CACHE / 'solve_meta.json', 'w') as _f:
    _json.dump(solve_meta, _f, indent=2)

print('Solution saved to cache:')
print(f'  {CACHE}/selected_gw.parquet')
print(f'  {CACHE}/assignments.parquet')
print(f'  {CACHE}/solve_meta.json')

---

## 6.10 — Solution Analysis and Gateway Characterization

Per-gateway and per-area reports; aggregate statistics.

In [ ]:
# ── 6.10.1  Load solution and supporting data ──────────────────────────────────
selected_gw_df = pd.read_parquet(CACHE / 'selected_gw.parquet')
assignments_df = pd.read_parquet(CACHE / 'assignments.parquet')
area_metrics   = pd.read_csv(DATA_T6 / 'area_metrics_phase2.csv')
area_ids       = area_metrics['area_id'].values
area_id_to_idx = {aid: i for i, aid in enumerate(area_ids)}
m_a            = area_metrics['m_a'].values
T_a            = area_metrics['external_throughput_ktons'].values
T_bar          = float(T_a.mean())
Q_bar          = float(np.median(np.load(CACHE / 's_g.npy')))
cap_rhs        = np.load(CACHE / 'cap_rhs.npy')
s_g            = np.load(CACHE / 's_g.npy')
G_raw          = pd.read_parquet(CACHE / 'G_candidates.parquet')

# Area assignment for in-area flag
area_assign = pd.read_csv(DATA_T6 / 'area_assignment.csv', dtype={'fips': str})
area_assign['fips'] = area_assign['fips'].str.zfill(5)
fips_to_area_id = area_assign.set_index('fips')['area_id'].to_dict()

print('Inputs loaded for 6.10')

In [ ]:
# ── 6.10.2  Per-gateway report ────────────────────────────────────────────────
# Add distance to primary area centroid (nearest assigned area)
transformer_inv = Transformer.from_crs('EPSG:4326', 'EPSG:9311', always_xy=True)

hub_rows = []
for _, gw in selected_gw_df.iterrows():
    g_idx_val  = int(gw['g_idx'])
    gx, gy = G_raw.loc[g_idx_val, 'x_m'], G_raw.loc[g_idx_val, 'y_m']
    _raw = gw['areas_served']
    areas_srv = list(_raw) if hasattr(_raw, '__iter__') and not isinstance(_raw, str) else []

    # Distance to each assigned area centroid
    area_dists = []
    c_hats_asgn = []
    for aid in areas_srv:
        a_idx_val = area_id_to_idx[aid]
        ax = area_metrics.loc[a_idx_val, 'centroid_x_m']
        ay = area_metrics.loc[a_idx_val, 'centroid_y_m']
        dist_m = math.sqrt((gx - ax)**2 + (gy - ay)**2)
        area_dists.append(round(dist_m / 1609.34, 2))

        # c_hat for this assignment
        row = assignments_df[(assignments_df['g_idx'] == g_idx_val) &
                             (assignments_df['area_id'] == aid)]
        c_hats_asgn.append(float(row['c_hat'].values[0]) if len(row) else float('nan'))

    hub_rows.append({
        'candidate_id':            gw['candidate_id'],
        'facility_name':           gw['facility_name'],
        'city':                    gw['city'],
        'state':                   gw['source_state'],
        'areas_served':            areas_srv,
        'n_areas_served':          len(areas_srv),
        'sqft':                    int(gw['usable_available_space_sf']),
        'dist_to_area_centroid_mi':area_dists,
        'c_hat_per_area':          c_hats_asgn,
    })

hub_report = pd.DataFrame(hub_rows)
print(f'hub_report: {len(hub_report):,} gateways')
print(hub_report[['candidate_id','facility_name','city','state','n_areas_served','sqft']].head(10).to_string(index=False))

In [ ]:
# ── 6.10.3  Per-area report ───────────────────────────────────────────────────
area_rows = []
for a_idx_val, row_am in area_metrics.iterrows():
    aid = row_am['area_id']

    # All assignments for this area
    asgn = assignments_df[assignments_df['area_id'] == aid]
    gw_cids       = asgn['candidate_id'].tolist()
    total_sqft    = float(s_g[asgn['g_idx'].values].sum()) if len(asgn) else 0.0
    rhs           = float(cap_rhs[a_idx_val])
    cap_slack     = total_sqft - rhs
    n_selected    = len(asgn)

    # Demand-weighted distance to nearest gateway
    # Load county coords for this area
    county_mask = area_assign['area_id'] == aid
    area_fips   = area_assign[county_mask]['fips'].values

    # Flag out-of-area gateways
    n_out_of_area = 0
    for _, arow in asgn.iterrows():
        gw_county = G_raw.loc[int(arow['g_idx']), 'county_fips']
        if str(gw_county).zfill(5) not in area_fips:
            n_out_of_area += 1

    area_rows.append({
        'area_id':            aid,
        'region_id':          row_am['region_id'],
        'region_type':        row_am['region_type'],
        'm_a':                int(m_a[a_idx_val]),
        'n_selected':         n_selected,
        'gateway_cids':       gw_cids,
        'total_sqft':         round(total_sqft, 0),
        'rhs_sqft':           round(rhs, 0),
        'cap_slack_sqft':     round(cap_slack, 0),
        'n_out_of_area_gw':   n_out_of_area,
        'T_a_ktons':          round(float(row_am['external_throughput_ktons']), 1),
    })

area_report = pd.DataFrame(area_rows)
print(f'area_report: {len(area_report):,} areas')
print(area_report[['area_id','region_id','m_a','n_selected','total_sqft','rhs_sqft','cap_slack_sqft']].head(10).to_string(index=False))

In [ ]:
# ── 6.10.4  Aggregate statistics ──────────────────────────────────────────────
print('=== Aggregate Statistics — Task 6 Phase 2 ===')
print()
print(f'Total open gateways           : {len(hub_report):,}')
conc_counts = hub_report['n_areas_served'].value_counts().sort_index()
for n, cnt in conc_counts.items():
    print(f'  Gateways serving {n} area(s)  : {cnt:,}')

print()
print('Gateway sqft distribution:')
print(hub_report['sqft'].describe().apply(lambda x: f'{x:,.0f}'))

print()
print('Per-area capacity slack (sqft):')
print(area_report['cap_slack_sqft'].describe().apply(lambda x: f'{x:,.0f}'))
n_cap_fail = (area_report['cap_slack_sqft'] < 0).sum()
print(f'Areas failing capacity (slack<0): {n_cap_fail}')

print()
print(f'Areas with out-of-area gateway  : {(area_report["n_out_of_area_gw"] > 0).sum():,}')

print()
print('Gateways by region type:')
type_summary = (
    area_report.explode('gateway_cids')
    .dropna(subset=['gateway_cids'])
    .groupby('region_type')['gateway_cids'].nunique()
)
print(type_summary)

# Appalachian coverage status (regions 4 and 16)
print()
print('Appalachian areas (regions 4 & 16) — coverage check:')
appal = area_report[area_report['region_id'].isin([4, 16])]
print(appal[['area_id','m_a','n_selected','cap_slack_sqft']].to_string(index=False))

In [ ]:
# ── 6.10.5  Sanity checks — Task 6.10 ────────────────────────────────────────
# Every area must be covered by at least m_a gateways
under_covered = area_report[area_report['n_selected'] < area_report['m_a']]
if len(under_covered) > 0:
    print(f'WARNING: {len(under_covered)} areas under-covered:')
    print(under_covered[['area_id','m_a','n_selected']].to_string(index=False))
else:
    print(f'All 132 areas meet coverage target (n_selected ≥ m_a)  ✓')

# No gateway should serve more than 2 areas
over_concentrated = hub_report[hub_report['n_areas_served'] > 2]
assert len(over_concentrated) == 0, f'{len(over_concentrated)} gateways serve >2 areas'
print(f'No gateway serves more than 2 areas  ✓')

print()
print('6.10 sanity checks passed  ✓')

---

## 6.11 — Network Link Definition

Two link types:
1. **Area-to-regional-hub links**: each area's selected gateway(s) → regional hub(s) of parent region.
2. **Inter-area links**: adjacent areas within the same region (county adjacency graph).

In [ ]:
# ── 6.11.1  Area-to-regional-hub links ───────────────────────────────────────
# Load Task 5 selected hubs and hub-region assignments
t5_hubs         = pd.read_csv(DATA_T5 / 'selected_hubs.csv')
hub_region_asgn = pd.read_csv(DATA_T5 / 'hub_region_assignments.csv')

# region_id → regional hub candidate_id(s)
region_to_hubs: dict[int, list] = {}
for region_id, grp in hub_region_asgn.groupby('region_id'):
    region_to_hubs[int(region_id)] = grp['candidate_id'].tolist()

# Fallback: use selected_hubs.csv regions_served column if hub_region_assignments has different schema
if not region_to_hubs:
    for _, h in t5_hubs.iterrows():
        regions_served = h['regions_served']
        if isinstance(regions_served, str):
            import ast
            regions_served = ast.literal_eval(regions_served)
        if isinstance(regions_served, (list, tuple)):
            for r in regions_served:
                region_to_hubs.setdefault(int(r), []).append(h['candidate_id'])
        else:
            region_to_hubs.setdefault(int(regions_served), []).append(h['candidate_id'])

print(f'Regions with hub assignments: {len(region_to_hubs)}')

# Regional hub coords for distance computation
t5_hubs_xy = t5_hubs.set_index('candidate_id')[['latitude', 'longitude']]
t5_transformer = Transformer.from_crs('EPSG:4326', 'EPSG:9311', always_xy=True)
t5_x, t5_y = t5_transformer.transform(
    t5_hubs_xy['longitude'].values, t5_hubs_xy['latitude'].values
)
hub_xy_9311 = dict(zip(t5_hubs_xy.index, zip(t5_x.tolist(), t5_y.tolist())))

In [ ]:
# ── 6.11.1 (cont.)  Build gateway → regional-hub link table ──────────────────
area_to_hub_rows = []

for _, arow in area_report.iterrows():
    aid       = arow['area_id']
    region_id = int(arow['region_id'])
    T_a_val   = float(arow['T_a_ktons'])
    regional_hubs = region_to_hubs.get(region_id, [])

    # Selected gateways for this area
    asgn = assignments_df[assignments_df['area_id'] == aid]
    for _, ga_row in asgn.iterrows():
        gw_cid = ga_row['candidate_id']
        g_idx_val = int(ga_row['g_idx'])
        gx = float(G_raw.loc[g_idx_val, 'x_m'])
        gy = float(G_raw.loc[g_idx_val, 'y_m'])

        for rhub_cid in regional_hubs:
            rh_xy = hub_xy_9311.get(rhub_cid)
            if rh_xy is None:
                continue
            dist_m    = math.sqrt((gx - rh_xy[0])**2 + (gy - rh_xy[1])**2)
            dist_mi   = round(dist_m / 1609.34, 2)
            area_to_hub_rows.append({
                'gateway_candidate_id':       gw_cid,
                'regional_hub_candidate_id':  rhub_cid,
                'region_id':                  region_id,
                'area_id':                    aid,
                'distance_miles':             dist_mi,
                'external_throughput_ktons':  T_a_val,
            })

area_to_hub_df = pd.DataFrame(area_to_hub_rows)

# Verify no orphaned areas
areas_with_links = set(area_to_hub_df['area_id'].unique())
all_areas        = set(area_report['area_id'].unique())
orphaned         = all_areas - areas_with_links
if orphaned:
    print(f'WARNING: {len(orphaned)} orphaned areas with no hub link: {orphaned}')
else:
    print(f'All areas have at least one area-to-hub link  ✓')

print(f'Area-to-hub links table: {len(area_to_hub_df):,} rows')
print(area_to_hub_df.head(5).to_string(index=False))

In [ ]:
# ── 6.11.2  Inter-area links (intra-region county adjacency) ─────────────────
county_edges = pd.read_parquet(DATA_T3 / 'cache/ne_county_edges.parquet')
county_edges['fips_a'] = county_edges['fips_a'].astype(str).str.zfill(5)
county_edges['fips_b'] = county_edges['fips_b'].astype(str).str.zfill(5)

# Merge area_id for both endpoints
fips_to_area = area_assign.set_index('fips')['area_id'].to_dict()
fips_to_region = area_assign.set_index('fips')['region_id'].to_dict()

county_edges['area_a']   = county_edges['fips_a'].map(fips_to_area)
county_edges['area_b']   = county_edges['fips_b'].map(fips_to_area)
county_edges['region_a'] = county_edges['fips_a'].map(fips_to_region)
county_edges['region_b'] = county_edges['fips_b'].map(fips_to_region)

# Keep only cross-area edges in the same region
inter_area_edges = county_edges[
    (county_edges['area_a'].notna()) &
    (county_edges['area_b'].notna()) &
    (county_edges['area_a'] != county_edges['area_b']) &
    (county_edges['region_a'] == county_edges['region_b'])
].copy()

# Aggregate: one row per area-pair with shared_borders count
# Canonical pair order: sort to avoid duplicates
inter_area_edges['pair'] = inter_area_edges.apply(
    lambda r: tuple(sorted([r['area_a'], r['area_b']])), axis=1
)

# County-to-county throughput for cross-area flow: use throughput from ne_counties_prepared
counties_thru = gpd.read_file(DATA_T3 / 'derived/ne_counties_prepared.gpkg')[['fips','throughput']]
counties_thru['fips'] = counties_thru['fips'].astype(str).str.zfill(5)
fips_to_thru = counties_thru.set_index('fips')['throughput'].to_dict()

inter_area_edges['cross_flow'] = inter_area_edges.apply(
    lambda r: (fips_to_thru.get(r['fips_a'], 0.0) + fips_to_thru.get(r['fips_b'], 0.0)) / 2.0,
    axis=1
)

inter_area_link_df = (
    inter_area_edges
    .groupby('pair')
    .agg(
        shared_borders          = ('pair', 'count'),
        cross_area_flow_ktons   = ('cross_flow', 'sum'),
        region_id               = ('region_a', 'first'),
    )
    .reset_index()
)
inter_area_link_df['area_a'] = inter_area_link_df['pair'].apply(lambda p: p[0])
inter_area_link_df['area_b'] = inter_area_link_df['pair'].apply(lambda p: p[1])
inter_area_link_df = inter_area_link_df.drop(columns=['pair'])

print(f'Inter-area link table: {len(inter_area_link_df):,} rows')
print(inter_area_link_df.head(5).to_string(index=False))

In [ ]:
# ── 6.11.3  Save link tables ──────────────────────────────────────────────────
area_to_hub_df.to_csv(DATA_T6 / 'gateway_area_to_hub_links.csv', index=False)
inter_area_link_df.to_csv(DATA_T6 / 'gateway_inter_area_links.csv', index=False)

print(f'Saved: gateway_area_to_hub_links.csv  ({len(area_to_hub_df):,} rows)')
print(f'Saved: gateway_inter_area_links.csv   ({len(inter_area_link_df):,} rows)')

# Sanity: all areas covered by at least one area-to-hub link
assert set(area_to_hub_df['area_id'].unique()) == all_areas or len(orphaned) == 0, \
    'Orphaned areas remain — check region_to_hubs mapping'
print('6.11 sanity checks passed  ✓')

---

## 6.12 — Figures and Exports

Produce final maps and tabular outputs.

In [ ]:
# ── 6.12.1  Setup: NE county geometry and colour palette ─────────────────────
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
matplotlib.rcParams['figure.dpi'] = 120

counties_geo = gpd.read_file(DATA_T3 / 'derived/ne_counties_prepared.gpkg').to_crs('EPSG:4326')

# Map region_id to a colour
import matplotlib.cm as cm
region_ids_all = sorted(area_metrics['region_id'].unique())
cmap = cm.get_cmap('tab20', len(region_ids_all))
region_color = {rid: cmap(i) for i, rid in enumerate(region_ids_all)}

# Merge region_id onto county geometry for basemap colouring
area_assign_geo = pd.read_csv(DATA_T6 / 'area_assignment.csv', dtype={'fips': str})
area_assign_geo['fips'] = area_assign_geo['fips'].str.zfill(5)
counties_geo = counties_geo.merge(area_assign_geo[['fips', 'region_id']], on='fips', how='left')
counties_geo['color'] = counties_geo['region_id'].map(region_color)

print(f'County basemap loaded: {len(counties_geo):,} counties')

In [ ]:
# ── 6.12.2  Figure 1 — Gateway hub locations (revised) ───────────────────────
from matplotlib.lines import Line2D
import matplotlib.patheffects as pe
from shapely.geometry import box as _shapely_box

# Reload final selected gateways
selected_gw_df = pd.read_parquet(CACHE / 'selected_gw.parquet')
assignments_df = pd.read_parquet(CACHE / 'assignments.parquet')

# Add region_id to selected_gw_df (needed by downstream Cell 37 export)
_gw_region = {}
for _, _gw in selected_gw_df.iterrows():
    _raw = _gw['areas_served']
    _areas = list(_raw) if hasattr(_raw, '__iter__') and not isinstance(_raw, str) else []
    if _areas:
        _a_idx = area_id_to_idx[_areas[0]]
        _gw_region[_gw['g_idx']] = int(area_metrics.loc[_a_idx, 'region_id'])
    else:
        _gw_region[_gw['g_idx']] = -1
selected_gw_df['region_id'] = selected_gw_df['g_idx'].map(_gw_region)

# ── Area / region border layers ───────────────────────────────────────────
# Build fresh county GeoDataFrame with both area_id and region_id
area_assign_b = pd.read_csv(DATA_T6 / 'area_assignment.csv', dtype={'fips': str})
area_assign_b['fips'] = area_assign_b['fips'].str.zfill(5)
counties_base = gpd.read_file(DATA_T3 / 'derived/ne_counties_prepared.gpkg').to_crs('EPSG:4326')
counties_base['fips'] = counties_base['fips'].astype(str).str.zfill(5)
counties_base = counties_base.merge(
    area_assign_b[['fips', 'area_id', 'region_id']], on='fips', how='left'
)
area_borders   = counties_base.dissolve(by='area_id').reset_index()
region_borders = counties_base.dissolve(by='region_id').reset_index()

# ── Infrastructure layers ─────────────────────────────────────────────────
NE_BOX = _shapely_box(-80.5, 36.5, -66.9, 47.5)
T3_CACHE = DATA_T3 / 'cache'

_int_cache = T3_CACHE / 'ne_interstates.parquet'
if _int_cache.exists():
    interstates_ne = gpd.read_parquet(_int_cache)
else:
    _roads = gpd.read_file(DATA_T3 / 'raw/roads/North_American_Roads.shp')
    _ist   = _roads[(_roads['COUNTRY'] == 2) & (_roads['CLASS'] == 1)][['geometry']].copy()
    interstates_ne = gpd.clip(_ist, NE_BOX)
    interstates_ne.to_parquet(_int_cache)
print(f'Interstates: {len(interstates_ne):,} segments')

_rail_cache = T3_CACHE / 'ne_railroads.parquet'
if _rail_cache.exists():
    rails_ne = gpd.read_parquet(_rail_cache)
else:
    _rails_all = gpd.read_file(DATA_T3 / 'raw/rails/tl_2023_us_rails.shp')
    rails_ne   = gpd.clip(_rails_all[['geometry']], NE_BOX)
    rails_ne.to_parquet(_rail_cache)
print(f'Railways   : {len(rails_ne):,} segments')

interstates_ne = interstates_ne.to_crs('EPSG:4326')
rails_ne       = rails_ne.to_crs('EPSG:4326')

# ── Plot ──────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(1, 1, figsize=(14, 10))
ax.set_facecolor('#f0f4f8')

# County base fill
counties_base.plot(ax=ax, color='#e8edf2', edgecolor='none', linewidth=0, zorder=1)

# Area borders — thin, slate blue
area_borders.boundary.plot(ax=ax, edgecolor='#6b8caf', linewidth=0.5, zorder=2)

# Region borders — thicker, dark navy
region_borders.boundary.plot(ax=ax, edgecolor='#1a2e4a', linewidth=1.8, zorder=3)

# Railway — blue dashed (halo + line)
if len(rails_ne) > 0:
    rails_ne.plot(ax=ax, color='#bfdbfe', linewidth=2.2, linestyle='--', alpha=0.85, zorder=4)
    rails_ne.plot(ax=ax, color='#1d4ed8', linewidth=0.85, linestyle='--', alpha=0.90, zorder=5)

# Interstate — red (halo + line)
if len(interstates_ne) > 0:
    interstates_ne.plot(ax=ax, color='#fca5a5', linewidth=3.2, alpha=0.80, zorder=6)
    interstates_ne.plot(ax=ax, color='#b91c1c', linewidth=1.3, alpha=0.95, zorder=7)

# Gateway hubs — yellow circles, size ∝ sqft
gw_sizes = (selected_gw_df['usable_available_space_sf'] /
            selected_gw_df['usable_available_space_sf'].max() * 55 + 12)
ax.scatter(
    selected_gw_df['longitude'], selected_gw_df['latitude'],
    c='#fde047', s=gw_sizes, alpha=0.90, marker='o',
    edgecolors='#92400e', linewidths=0.5, zorder=9, label='Gateway hubs'
)

# Regional hubs — green stars
t5_hubs_ref = pd.read_csv(DATA_T5 / 'selected_hubs.csv')
regional_hub_pts = ax.scatter(
    t5_hubs_ref['longitude'], t5_hubs_ref['latitude'],
    c='#16a34a', s=130, marker='*', edgecolors='#14532d',
    linewidths=0.8, zorder=10, label='Regional hubs'
)
regional_hub_pts.set_path_effects([pe.withStroke(linewidth=1.8, foreground='white')])

# Legend
legend_elements = [
    Line2D([0],[0], marker='o', color='w', markerfacecolor='#fde047',
           markeredgecolor='#92400e', markersize=9, label='Gateway hub (size ∝ sqft)'),
    Line2D([0],[0], marker='*', color='w', markerfacecolor='#16a34a',
           markeredgecolor='#14532d', markersize=12, label='Regional hub'),
    Line2D([0],[0], color='#1a2e4a', linewidth=2.0, label='Region border'),
    Line2D([0],[0], color='#6b8caf', linewidth=0.8, label='Area border'),
    Line2D([0],[0], color='#b91c1c', linewidth=1.5, label='Interstate highway'),
    Line2D([0],[0], color='#1d4ed8', linewidth=1.0, linestyle='--', label='Railway'),
]
ax.legend(handles=legend_elements, loc='lower right', fontsize=8.5,
          framealpha=0.92, edgecolor='#334155')

ax.set_title('Figure 1 — Gateway Hub Locations\n'
             'Yellow = gateway hubs (size ∝ sqft); Green \u2605 = regional hubs; '
             'Dark border = region; Light border = area', fontsize=11)
ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
ax.set_aspect('equal')
plt.tight_layout()

fig.savefig(FIG_DIR / 'fig_gateway_hub_locations.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Figure 1 saved: {FIG_DIR}/fig_gateway_hub_locations.png')


In [ ]:
# ── 6.12.3  Figure 2 — Gateway hub network links ──────────────────────────────
area_to_hub_df  = pd.read_csv(DATA_T6 / 'gateway_area_to_hub_links.csv')
inter_area_df   = pd.read_csv(DATA_T6 / 'gateway_inter_area_links.csv')

# Build gateway coord lookup
gw_latlon = selected_gw_df.set_index('candidate_id')[['latitude','longitude']].to_dict(orient='index')
hub_latlon = t5_hubs_ref.set_index('candidate_id')[['latitude','longitude']].to_dict(orient='index')

# Area centroid lookup (lat/lon via back-projection)
inv_transformer = Transformer.from_crs('EPSG:9311', 'EPSG:4326', always_xy=True)
area_lon, area_lat = inv_transformer.transform(
    area_metrics['centroid_x_m'].values,
    area_metrics['centroid_y_m'].values
)
area_latlon = {aid: (float(area_lat[i]), float(area_lon[i])) for i, aid in enumerate(area_ids)}

fig, ax = plt.subplots(1, 1, figsize=(14, 10))
ax.set_facecolor('#eef4fb')
counties_geo.plot(ax=ax, color='#eef2f7', edgecolor='white', linewidth=0.25, zorder=1)
region_borders.boundary.plot(ax=ax, edgecolor='#334155', linewidth=1.0, alpha=0.8, zorder=2)

# Normalize flow for link thickness
max_flow = max(float(area_to_hub_df['external_throughput_ktons'].max()), 1.0)

# Area-to-hub links (colored by region)
for _, lrow in area_to_hub_df.iterrows():
    gw_info  = gw_latlon.get(lrow['gateway_candidate_id'])
    rh_info  = hub_latlon.get(lrow['regional_hub_candidate_id'])
    if gw_info is None or rh_info is None:
        continue
    col  = region_color.get(int(lrow['region_id']), '#888888')
    lw   = 0.8 + 3.8 * (float(lrow['external_throughput_ktons']) / max_flow)
    ax.plot(
        [gw_info['longitude'], rh_info['longitude']],
        [gw_info['latitude'],  rh_info['latitude']],
        color='white', linewidth=lw + 1.4, alpha=0.35, zorder=3
    )
    ax.plot(
        [gw_info['longitude'], rh_info['longitude']],
        [gw_info['latitude'],  rh_info['latitude']],
        color=col, linewidth=lw, alpha=0.82, zorder=4
    )

# Inter-area links (gray)
max_ia_flow = inter_area_df['cross_area_flow_ktons'].max() if len(inter_area_df) > 0 else 1.0
for _, irow in inter_area_df.iterrows():
    lc_a = area_latlon.get(irow['area_a'])
    lc_b = area_latlon.get(irow['area_b'])
    if lc_a is None or lc_b is None:
        continue
    lw = 0.35 + 2.0 * (float(irow['cross_area_flow_ktons']) / max_ia_flow)
    ax.plot(
        [lc_a[1], lc_b[1]], [lc_a[0], lc_b[0]],
        color='white', linewidth=lw + 0.8, alpha=0.25, zorder=2
    )
    ax.plot(
        [lc_a[1], lc_b[1]], [lc_a[0], lc_b[0]],
        color='#64748b', linewidth=lw, alpha=0.45, linestyle='--', zorder=3
    )

# Gateway and regional hub markers
gw_sizes = (selected_gw_df['usable_available_space_sf'] /
            selected_gw_df['usable_available_space_sf'].max() * 32 + 18)
ax.scatter(selected_gw_df['longitude'], selected_gw_df['latitude'],
           c='#fde047', s=gw_sizes, alpha=0.92, edgecolors='#92400e',
           linewidths=0.45, zorder=5, label='Gateway hubs')
regional_hub_network_pts = ax.scatter(t5_hubs_ref['longitude'], t5_hubs_ref['latitude'],
           c='#2563eb', s=145, marker='*', edgecolors='#1e3a8a',
           linewidths=0.9, zorder=6, label='Regional hubs')
regional_hub_network_pts.set_path_effects([pe.withStroke(linewidth=2.0, foreground='white')])

# Legend items for links
legend_elements = [
    Line2D([0],[0], color='#334155', linewidth=2.6, label='Area→Hub links (colored by region)'),
    Line2D([0],[0], color='#64748b', linewidth=1.6, linestyle='--', label='Inter-area links'),
    Line2D([0],[0], marker='o', color='w', markerfacecolor='#fde047',
           markeredgecolor='#92400e', markersize=8, label='Gateway hubs'),
    Line2D([0],[0], marker='*', color='w', markerfacecolor='#2563eb',
           markeredgecolor='#1e3a8a', markersize=12, label='Regional hubs'),
]

ax.set_title('Figure 2 — Gateway Hub Network\n'
             'Blue \u2605 = regional hubs; colored area→hub links and dashed inter-area links; width ∝ throughput',
             fontsize=11)
ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
ax.legend(handles=legend_elements, loc='lower right', fontsize=8.5,
          framealpha=0.94, edgecolor='#334155')
ax.set_aspect('equal')
plt.tight_layout()

fig.savefig(FIG_DIR / 'fig_gateway_hub_network.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Figure 2 saved: {FIG_DIR}/fig_gateway_hub_network.png')

In [ ]:
# ── 6.12.4  Tabular exports ────────────────────────────────────────────────────
# gateway_selected.csv — all selected gateways with full characterisation
gw_export = selected_gw_df[[
    'candidate_id','facility_name','city','source_state',
    'latitude','longitude','usable_available_space_sf',
    'areas_served','n_areas_served','region_id',
]].copy()
gw_export['areas_served'] = gw_export['areas_served'].apply(
    lambda x: '|'.join(list(x)) if hasattr(x, '__iter__') and not isinstance(x, str) else str(x)
)
gw_export.to_csv(DATA_T6 / 'gateway_selected.csv', index=False)
print(f'gateway_selected.csv        : {len(gw_export):,} rows')

# gateway_area_assignments.csv — full (g, a) pair list with ĉ_ga
asgn_export = assignments_df[[
    'candidate_id','area_id','c_hat',
    'facility_name','city','source_state','usable_available_space_sf',
]].copy()
asgn_export.to_csv(DATA_T6 / 'gateway_area_assignments.csv', index=False)
print(f'gateway_area_assignments.csv: {len(asgn_export):,} rows')

# area_metrics_phase2.csv — extended with solution columns
area_metrics_out = area_metrics.copy()
area_report_merge = area_report[[
    'area_id','n_selected','total_sqft','rhs_sqft','cap_slack_sqft'
]].rename(columns={
    'n_selected':      'n_selected_gateways',
    'total_sqft':      'total_assigned_sqft',
    'rhs_sqft':        'capacity_rhs_sqft',
    'cap_slack_sqft':  'capacity_slack_sqft',
})
area_metrics_out = area_metrics_out.merge(area_report_merge, on='area_id', how='left')
area_metrics_out.to_csv(DATA_T6 / 'area_metrics_phase2.csv', index=False)
print(f'area_metrics_phase2.csv     : {len(area_metrics_out):,} rows  (overwritten with solution columns)')

print()
print('All tabular exports complete.')

In [ ]:
# ── 6.12.5  Final sanity check ────────────────────────────────────────────────
from pathlib import Path

expected_files = [
    DATA_T6 / 'gateway_selected.csv',
    DATA_T6 / 'gateway_area_assignments.csv',
    DATA_T6 / 'gateway_area_to_hub_links.csv',
    DATA_T6 / 'gateway_inter_area_links.csv',
    DATA_T6 / 'area_metrics_phase2.csv',
    FIG_DIR  / 'fig_gateway_hub_locations.png',
    FIG_DIR  / 'fig_gateway_hub_network.png',
]

all_ok = True
for fp in expected_files:
    exists = Path(fp).exists()
    size   = Path(fp).stat().st_size if exists else 0
    mark   = '✓' if exists and size > 0 else '✗'
    print(f'  {mark}  {fp.name:<45} {size:>10,} bytes')
    if not (exists and size > 0):
        all_ok = False

print()
if all_ok:
    print('Task 6 Phase 2 complete  ✓  All outputs verified.')
else:
    print('Some outputs missing — re-run upstream cells.')

---

## Milestone — Task 6 Phase 2 complete

**Subtasks completed:** 6.7 · 6.8 · 6.9 · 6.10 · 6.11 · 6.12

**Key outputs:**

| File | Description |
|------|-------------|
| `Data/Task6/gateway_selected.csv` | All selected gateway hubs with characterisation |
| `Data/Task6/gateway_area_assignments.csv` | Full (g, a) assignment list with ĉ_ga |
| `Data/Task6/gateway_area_to_hub_links.csv` | Area → regional hub network links |
| `Data/Task6/gateway_inter_area_links.csv` | Intra-region inter-area adjacency links |
| `Data/Task6/area_metrics_phase2.csv` | Area metrics extended with MIP solution columns |
| `Data/Task6/figures/fig_gateway_hub_locations.png` | Hub location map |
| `Data/Task6/figures/fig_gateway_hub_network.png` | Hub network link map |

**Next:** Task 7 — Multi-Tier Integration